# Analysis

**Hypothesis**: Within each annotated cardiac population, spatially localized subpopulations exist whose transcriptional programs are systematically associated with gradients in sample-level tissue purity, reflecting differential vulnerability or adaptation to microenvironmental quality during heart development.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_anon.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Within each annotated cardiac population, spatially localized subpopulations exist whose transcriptional programs are systematically associated with gradients in sample-level tissue purity, reflecting differential vulnerability or adaptation to microenvironmental quality during heart development.

## Steps:
- Inspect and clean available metadata (Populations, Purity, Sample_ID, Batch, UMI Count, Complexity, leiden), ensuring appropriate dtypes (categorical vs numeric), summarizing Purity and QC metric distributions overall and within Population × Sample_ID strata, and identifying well-powered groups (e.g., ≥200 cells per Population and per Population × Sample_ID).
- Within each sufficiently large Population (and, where possible, within Population × Sample_ID groups), test for Purity–spatial structure using the 2D spatial coordinates in obsm['spatial'] (and optionally X_umap as a non-spatial control) by correlating Purity with x and y, and by computing simple spatial trend statistics, while also comparing UMI Count and Complexity between high- vs low-Purity cells per Population using Wilcoxon rank-sum tests and reporting p-values to rule out trivial QC confounding.
- For major Populations that meet cell-count/QC criteria, perform strictly within-Population unsupervised subclustering using the existing neighborhood graph/embeddings (restricting to each Population at a time), derive subcluster labels, and then assess whether subcluster composition and Purity distributions differ across Sample_IDs to separate sample-driven from shared biological structure.
- Within each qualifying Population, run differential expression analysis using sc.tl.rank_genes_groups (method='wilcoxon') separately (i) between high- vs low-Purity cells (e.g., top vs bottom Purity quartile) and (ii) between Purity-enriched vs Purity-depleted subclusters, performing tests within-Population and, where powered, within-Population × Sample_ID, and summarize top genes and statistics in text tables.
- Construct data-driven Purity-associated gene signatures per Population from high- vs low-Purity DE results (e.g., top up- and down-regulated genes), score each cell with sc.tl.score_genes, and model signature scores as a function of Purity using linear regression with UMI Count, Complexity, and Sample_ID (or per-sample regressions) as covariates, reporting coefficients and p-values.
- For Populations showing robust Purity associations, summarize which gene programs and subclusters track Purity gradients and evaluate cross-sample consistency by comparing the sign and magnitude of within-Sample_ID correlations between Purity and signature scores (and, if needed, meta-analyzing via Fisher’s z), reporting a text-based synthesis of consistent vs sample-specific patterns.


## Inspect and lightly clean the AnnData metadata by enforcing appropriate dtypes for key categorical variables, summarizing Purity and QC metrics overall and within Population × Sample_ID strata, and quantifying cell counts per group to identify well-powered combinations for downstream Purity-gradient analyses.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# Basic AnnData overview
print("AnnData shape (cells x genes):", adata.n_obs, "x", adata.n_vars)

# Ensure key categorical columns are of type 'category' for efficient grouping
for col in ["Populations", "Sample_ID", "Batch", "leiden"]:
    if col in adata.obs.columns and not pd.api.types.is_categorical_dtype(adata.obs[col]):
        adata.obs[col] = adata.obs[col].astype("category")

# List obs columns and their types
print("\n.obs columns and dtypes:")
print(adata.obs.dtypes)

# Show first few rows of key metadata
meta_cols = [col for col in ["Sample_ID", "Batch", "Populations", "Purity", "UMI Count", "Complexity", "leiden"] if col in adata.obs.columns]
print("\nHead of key metadata columns:")
print(adata.obs[meta_cols].head())

# Explicitly confirm numeric dtypes and missingness for key continuous covariates
for col in ["Purity", "UMI Count", "Complexity"]:
    if col in adata.obs.columns:
        series = adata.obs[col]
        print(f"\n{col} summary and checks:")
        print("  dtype:", series.dtype)
        print("  describe():")
        print(series.describe())
        n_missing = series.isna().sum()
        print(f"  missing values: {n_missing}")

# Distribution of Purity overall and stratified
if "Purity" in adata.obs.columns:
    if "Populations" in adata.obs.columns:
        print("\nPurity by Populations (mean, std, count):")
        purity_by_pop = (
            adata.obs.groupby("Populations")["Purity"]
            .agg(["mean", "std", "count"])
            .sort_values("mean", ascending=False)
        )
        print(purity_by_pop)

    if "Sample_ID" in adata.obs.columns:
        print("\nPurity by Sample_ID (mean, std, count):")
        purity_by_sample = (
            adata.obs.groupby("Sample_ID")["Purity"]
            .agg(["mean", "std", "count"])
            .sort_values("mean", ascending=False)
        )
        print(purity_by_sample)

    if "Populations" in adata.obs.columns and "Sample_ID" in adata.obs.columns:
        print("\nPurity by Populations and Sample_ID (mean, std, count):")
        purity_by_pop_sample = (
            adata.obs
            .groupby(["Populations", "Sample_ID"])["Purity"]
            .agg(["mean", "std", "count"])
            .sort_values(["Populations", "Sample_ID"])
        )
        print(purity_by_pop_sample)

# Cell counts per Population, Sample_ID, and their combination to assess power
if "Populations" in adata.obs.columns:
    print("\nCell counts per Population:")
    counts_pop = adata.obs["Populations"].value_counts()
    print(counts_pop)

if "Sample_ID" in adata.obs.columns:
    print("\nCell counts per Sample_ID:")
    counts_sample = adata.obs["Sample_ID"].value_counts()
    print(counts_sample)

if "Populations" in adata.obs.columns and "Sample_ID" in adata.obs.columns:
    print("\nTop 20 Population × Sample_ID combinations by cell count:")
    pop_sample_counts = (
        adata.obs.groupby(["Populations", "Sample_ID"]).size()
        .rename("n_cells")
        .sort_values(ascending=False)
    )
    print(pop_sample_counts.head(20))

# Check ranges and basic stats for UMI Count and Complexity
for col in ["UMI Count", "Complexity"]:
    if col in adata.obs.columns:
        print(f"\nSummary of {col}:")
        print(adata.obs[col].describe())
        n_missing = adata.obs[col].isna().sum()
        print(f"Missing values in {col}: {n_missing}")

AnnData shape (cells x genes): 228635 x 238

.obs columns and dtypes:
Sample_ID      category
Batch          category
UMI Count       float64
leiden         category
Complexity        int64
Populations    category
Purity          float64
dtype: object

Head of key metadata columns:
           Sample_ID    Batch Populations    Purity  UMI Count  Complexity  \
cell_id                                                                      
6-R77_4C4    R77_4C4  R77_4C4          PG  0.544534       86.0           8   
8-R77_4C4    R77_4C4  R77_4C4          PG  0.625984      148.0           8   
9-R77_4C4    R77_4C4  R77_4C4          PG  0.583665      100.0           8   
10-R77_4C4   R77_4C4  R77_4C4          PG  0.766393       70.0           9   
12-R77_4C4   R77_4C4  R77_4C4          PG  0.596838       63.0           8   

           leiden  
cell_id            
6-R77_4C4       6  
8-R77_4C4       6  
9-R77_4C4       6  
10-R77_4C4      6  
12-R77_4C4      6  

Purity summary and checks:
  

### Agent Interpretation

The metadata/QC inventory looks solid and is a good foundation for the rest of the plan. Here’s how these results inform the next steps and the hypothesis.

1. Overall data quality / metadata readiness  
- Purity, UMI Count, and Complexity are all numeric with no missing values, which is ideal for downstream modeling and covariate adjustment.  
- The ranges look reasonable for MERFISH-style data: moderate UMI and complexity, Purity spanning 0.13–1.0 with a broad distribution (sd ~0.15). This gives you room to define meaningful high-/low-Purity strata and to detect gradients.  
- Key grouping variables (Populations, Sample_ID, Batch, leiden) are already categorical, which will make the stratified and within-population analyses straightforward.

2. Power assessment for Population and Population × Sample_ID  
- At the Population level, you have excellent power: almost all populations have >200 cells; many have thousands to tens of thousands. The smallest (PAA) still has >1000 cells overall. So nearly all populations can be included in population-level analyses.  
- At the Population × Sample_ID level, the top combinations have thousands of cells. You will be able to do within-Population × Sample_ID analyses for the larger pairs (e.g., PA×any sample; PB/PC/PD/PE/PI/PH/PG×several samples).  
- You should still explicitly set and apply a cutoff (e.g., ≥200 cells per Populations × Sample_ID) to decide where you can safely do:  
  - spatial-Purity trend tests,  
  - high vs low Purity DE,  
  - subclustering comparisons.

3. Structure of Purity across populations and samples  
- Purity varies substantially across Populations (means ~0.39–0.70). Some populations (PB, PS, PG, PR, PI, PM) are on the higher-Purity end; others (PJ, PL, PD, PC, PA) are lower. This variation is promising for your hypothesis: it means “microenvironmental quality” (as proxied by Purity) is not homogeneous and may be differentially experienced by different cell types.  
- Within-population variability (std) is nontrivial in almost all clusters (often ~0.1–0.16, PR as high as 0.22). This intra-population spread is exactly what you need to look for within-Population Purity gradients and subpopulations.  
- Across Sample_IDs, Purity means are very similar (~0.498–0.51). That suggests the Purity metric isn’t grossly different between samples, reducing the risk that your findings are dominated by global per-sample technical artifacts. Still, within each Populations × Sample_ID, the mean and spread will matter; you already have those summarized and they look reasonably stable but somewhat variable, which is useful signal.

4. Populations that look particularly promising for the hypothesis  
Because your hypothesis is about within-population substructure linked to Purity and spatial gradients, you want populations that have: large n, substantial Purity variance, and broad sampling across all three Sample_IDs.

From the summaries, strong candidates include:  
- High-Purity means with good counts and spread:  
  - PB (n ~20k, mean ~0.70, sd ~0.12)  
  - PS (n ~4.6k, mean ~0.66, sd ~0.11)  
  - PG (n ~11.6k, mean ~0.64, sd ~0.17)  
  - PR (n ~4.7k, mean ~0.62, sd ~0.22)  
  - PI (n ~10.4k, mean ~0.61, sd ~0.11)  
- Mid-range Purity but large size and decent variance (likely to harbor mixed microenvironments):  
  - PA (n ~30k, mean ~0.47, sd ~0.09)  
  - PC, PD, PE, PF, PH, PM, PN (all 7–17k cells, means around 0.42–0.58, sd ~0.10–0.15)  

These will be your primary “discovery” populations to explore spatial Purity gradients, subclusters, and transcriptional programs. Smaller populations (e.g., PX, PY, PZ, PAA) could be explored exploratorily but will have less power, especially per-sample.

5. Implications for QC confounding and later modeling  
- UMI Count and Complexity have substantial variation but no missingness. Before interpreting biological Purity gradients, you’ll need to test whether Purity is strongly correlated with these QC metrics within each Population (and per Sample_ID where possible). That aligns with your next plan steps: compare UMI/Complexity between high- and low-Purity cells using Wilcoxon tests, and later include them as covariates in regression models of Purity-associated signatures.  
- The distributions (median UMI ~386, IQR ~237–583; Complexity median ~10, IQR ~8–12) suggest moderate but not extreme variation. This is good—it’s unlikely that QC will completely dominate signal, but you must still check whether low-Purity cells systematically have fewer UMIs/genes in specific populations.

6. Recommendations for the immediate next steps (step 2 of your plan)  
Given these results, you’re ready to:

a) Spatial–Purity trend tests  
- Restrict to populations with n ≥ 500–1000 overall and, for within-sample analyses, Populations × Sample_ID with n ≥ 200. Explicitly compute:  
  - Correlation of Purity with spatial x and y (within each Population, and within each Population × Sample_ID when powered).  
  - As a non-spatial control: correlation of Purity with UMAP coordinates (if present in obsm['X_umap']) to separate purely transcriptomic gradients from physical-spatial ones.  
- Focus initially on the high-variance/high-n populations (PB, PG, PR, PI, PF, PH, PA, PC, PD, PE). They are most likely to exhibit continuous Purity gradients or spatially localized high-/low-Purity regions.  

b) QC checks stratified by Purity  
- Within each population, partition cells by Purity (e.g., quartiles or top/bottom 30%).  
- For each population (and for larger Populations × Sample_ID):  
  - Perform Wilcoxon rank-sum tests of UMI Count and Complexity between high- vs low-Purity groups.  
  - Flag populations where Purity is almost entirely confounded with severe UMI/Complexity differences; in those, any DE results must be interpreted cautiously and require stronger QC adjustment.  
- Summarize which populations show modest QC–Purity coupling (more trustworthy for biological interpretation) vs strong coupling (where you may need to downweight or treat findings as exploratory).

c) Prepare for subclustering and DE  
- Based on the cell counts per population, plan to do within-Population subclustering for at least: PA–PH, PM, PN, PQ, PR, PS, PT, maybe down to PU/PV/PW depending on interest.  
- For Populations × Sample_ID combinations with large n, note them as prime targets for “controlled” analyses isolating within-sample structure (e.g., PA×R78_4C15; PB×each sample; PC/PD/PE×each sample).

7. How this bears on the hypothesis so far  
- The metadata/QC landscape is compatible with your hypothesis: there is clear intra-population variation in Purity across many well-powered populations, without gross per-sample Purity shifts that would trivially explain everything.  
- You have the statistical power and structure (multiple samples, large cell numbers, moderate QC variation) to robustly test whether Purity associates with:  
  - spatial coordinates,  
  - subcluster structure,  
  - and distinct transcriptional programs within each annotated population.  

The hypothesis is not yet validated or refuted by these results alone, but the data structure is highly favorable for testing it. The key next steps are: (1) quantify Purity–spatial coupling within each major population while checking QC confounding, and (2) then move toward subclustering/DE in the best-behaved, well-powered populations that show nontrivial Purity gradients.

## Next Steps
Step 1: Within each sufficiently large Population (and, where powered, Population × Sample_ID; e.g., n ≥ 200 cells), quantify the association between Purity and spatial coordinates (obsm['spatial']) versus UMAP coordinates (obsm['X_umap']) using Pearson and Spearman correlations, summarize effect sizes and unadjusted p-values, and define explicit criteria (e.g., |rho_spatial|max − |rho_umap|max ≥ 0.1 with FDR-adjusted spatial p < 0.05) to flag Populations with genuine spatial–Purity structure not mirrored in UMAP space.
Step 2: For flagged Populations, assess QC confounding by (i) computing correlations of Purity with UMI Count and Complexity and (ii) comparing these QC metrics between high- and low-Purity strata (e.g., top vs bottom Purity quartile) using Wilcoxon rank-sum tests, reporting effect sizes and FDR-adjusted p-values and retaining Populations where QC differences are modest.
Step 3: In Populations with spatially structured but only modestly QC-confounded Purity, perform within-Population subclustering using the existing neighbor graph/embeddings (restricting to that Population without recomputing global neighbors), assign subcluster labels, and test for differences in Purity distributions across subclusters via Kruskal–Wallis tests overall and, when powered, within each Sample_ID.
Step 4: Within these selected Populations, perform differential expression with sc.tl.rank_genes_groups (method='wilcoxon') comparing high- vs low-Purity cells and, separately, Purity-enriched vs Purity-depleted subclusters, both at the Population level and in within-sample sensitivity analyses where powered, and construct per-Population Purity-up and Purity-down signatures from top DE genes based on adjusted p-values and effect sizes.
Step 5: Score each cell for the Purity-up and Purity-down signatures using sc.tl.score_genes and model these scores via linear regression against Purity while adjusting for UMI Count, Complexity, and Sample_ID (and, where feasible, in separate within-sample models), summarizing regression coefficients and p-values to quantify how consistently Purity-associated programs track microenvironmental tissue-quality gradients across samples.

## This code quantifies how strongly Purity associates with spatial versus UMAP coordinates within each Population and Population×Sample_ID stratum, applies an explicit FDR-based criterion to flag Populations with genuine spatial–Purity structure not mirrored in UMAP space, and stores both detailed and summarized results in adata.uns for downstream QC, subclustering, and DE analyses.

In [ ]:

import numpy as np
import pandas as pd
from scipy import stats

# Ensure required embeddings are present
assert 'spatial' in adata.obsm, "Spatial coordinates not found in adata.obsm['spatial']"
if 'X_umap' not in adata.obsm:
    raise ValueError("X_umap embedding not found in adata.obsm['X_umap']; required as a non-spatial control")

# Extract coordinates
spatial = adata.obsm['spatial']
umap = adata.obsm['X_umap']
adata.obs['spatial_x'] = spatial[:, 0]
adata.obs['spatial_y'] = spatial[:, 1]
adata.obs['umap_x'] = umap[:, 0]
adata.obs['umap_y'] = umap[:, 1]

# Define groupings and power thresholds
min_cells_pop = 200
min_cells_pop_sample = 200

pop_col = 'Populations'
sample_col = 'Sample_ID'

if pop_col not in adata.obs.columns:
    raise ValueError(f"Required column '{pop_col}' not found in adata.obs")
if sample_col not in adata.obs.columns:
    raise ValueError(f"Required column '{sample_col}' not found in adata.obs")
if 'Purity' not in adata.obs.columns:
    raise ValueError("Required column 'Purity' not found in adata.obs")

# Ensure Purity is numeric
adata.obs['Purity'] = pd.to_numeric(adata.obs['Purity'], errors='coerce')

# Helper to compute correlations with both Pearson and Spearman
def corr_summary(x, y):
    """Return Pearson and Spearman r and p-values between x and y."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    if x_valid.size < 3:
        return {
            'pearson_r': np.nan,
            'pearson_p': np.nan,
            'spearman_rho': np.nan,
            'spearman_p': np.nan,
            'n': int(x_valid.size),
        }
    pr, pp = stats.pearsonr(x_valid, y_valid)
    sr, sp = stats.spearmanr(x_valid, y_valid)
    return {
        'pearson_r': pr,
        'pearson_p': pp,
        'spearman_rho': sr,
        'spearman_p': sp,
        'n': int(x_valid.size),
    }

results = []

# Per-Population correlations
for pop, idx in adata.obs.groupby(pop_col).groups.items():
    # ensure indices are integers
    idx = np.asarray(idx)
    if not np.issubdtype(idx.dtype, np.integer):
        idx = idx.astype(int)
    n_cells = len(idx)
    if n_cells < min_cells_pop:
        continue
    sub = adata.obs.iloc[idx]
    purity = sub['Purity'].values

    # Spatial correlations (x,y vs Purity)
    csx = corr_summary(sub['spatial_x'].values, purity)
    csy = corr_summary(sub['spatial_y'].values, purity)

    # UMAP (non-spatial) correlations
    cux = corr_summary(sub['umap_x'].values, purity)
    cuy = corr_summary(sub['umap_y'].values, purity)

    results.append({
        'level': 'population',
        'population': str(pop),
        'sample_id': 'ALL',
        'n_cells': n_cells,
        'spatial_x_pearson_r': csx['pearson_r'],
        'spatial_x_pearson_p': csx['pearson_p'],
        'spatial_x_spearman_rho': csx['spearman_rho'],
        'spatial_x_spearman_p': csx['spearman_p'],
        'spatial_y_pearson_r': csy['pearson_r'],
        'spatial_y_pearson_p': csy['pearson_p'],
        'spatial_y_spearman_rho': csy['spearman_rho'],
        'spatial_y_spearman_p': csy['spearman_p'],
        'umap_x_pearson_r': cux['pearson_r'],
        'umap_x_pearson_p': cux['pearson_p'],
        'umap_x_spearman_rho': cux['spearman_rho'],
        'umap_x_spearman_p': cux['spearman_p'],
        'umap_y_pearson_r': cuy['pearson_r'],
        'umap_y_pearson_p': cuy['pearson_p'],
        'umap_y_spearman_rho': cuy['spearman_rho'],
        'umap_y_spearman_p': cuy['spearman_p'],
    })

# Per-Population × Sample_ID correlations (when powered)
for (pop, sample), idx in adata.obs.groupby([pop_col, sample_col]).groups.items():
    idx = np.asarray(idx)
    if not np.issubdtype(idx.dtype, np.integer):
        idx = idx.astype(int)
    n_cells = len(idx)
    if n_cells < min_cells_pop_sample:
        continue
    sub = adata.obs.iloc[idx]
    purity = sub['Purity'].values

    csx = corr_summary(sub['spatial_x'].values, purity)
    csy = corr_summary(sub['spatial_y'].values, purity)
    cux = corr_summary(sub['umap_x'].values, purity)
    cuy = corr_summary(sub['umap_y'].values, purity)

    results.append({
        'level': 'population_sample',
        'population': str(pop),
        'sample_id': str(sample),
        'n_cells': n_cells,
        'spatial_x_pearson_r': csx['pearson_r'],
        'spatial_x_pearson_p': csx['pearson_p'],
        'spatial_x_spearman_rho': csx['spearman_rho'],
        'spatial_x_spearman_p': csx['spearman_p'],
        'spatial_y_pearson_r': csy['pearson_r'],
        'spatial_y_pearson_p': csy['pearson_p'],
        'spatial_y_spearman_rho': csy['spearman_rho'],
        'spatial_y_spearman_p': csy['spearman_p'],
        'umap_x_pearson_r': cux['pearson_r'],
        'umap_x_pearson_p': cux['pearson_p'],
        'umap_x_spearman_rho': cux['spearman_rho'],
        'umap_x_spearman_p': cux['spearman_p'],
        'umap_y_pearson_r': cuy['pearson_r'],
        'umap_y_pearson_p': cuy['pearson_p'],
        'umap_y_spearman_rho': cuy['spearman_rho'],
        'umap_y_spearman_p': cuy['spearman_p'],
    })

results_df = pd.DataFrame(results)

# Compute simple multiple-testing-aware summaries at the population level
from statsmodels.stats.multitest import multipletests

if not results_df.empty:
    pop_df = results_df[results_df['level'] == 'population'].copy()
    # Max |rho| across x/y for spatial and UMAP (Spearman)
    pop_df['max_abs_spatial_rho'] = pop_df[['spatial_x_spearman_rho', 'spatial_y_spearman_rho']].abs().max(axis=1)
    pop_df['max_abs_umap_rho'] = pop_df[['umap_x_spearman_rho', 'umap_y_spearman_rho']].abs().max(axis=1)

    # FDR adjust minimum spatial p-value across x/y
    pop_df['min_spatial_p'] = pop_df[['spatial_x_spearman_p', 'spatial_y_spearman_p']].min(axis=1)
    _, pop_df['min_spatial_p_fdr'], _, _ = multipletests(pop_df['min_spatial_p'], method='fdr_bh')

    # Define a flag for strong spatial-Purity structure not mirrored in UMAP
    pop_df['strong_spatial_purity'] = (
        (pop_df['max_abs_spatial_rho'] - pop_df['max_abs_umap_rho'] >= 0.1) &
        (pop_df['min_spatial_p_fdr'] < 0.05)
    )

    # Print summary table
    summary_cols = [
        'population', 'n_cells',
        'spatial_x_spearman_rho', 'spatial_x_spearman_p',
        'spatial_y_spearman_rho', 'spatial_y_spearman_p',
        'umap_x_spearman_rho', 'umap_x_spearman_p',
        'umap_y_spearman_rho', 'umap_y_spearman_p',
        'max_abs_spatial_rho', 'max_abs_umap_rho',
        'min_spatial_p_fdr', 'strong_spatial_purity',
    ]
    pop_df = pop_df[summary_cols].sort_values('max_abs_spatial_rho', ascending=False)

    print("\n=== Per-Population spatial vs UMAP Purity correlations (Spearman, powered populations only) ===")
    print(pop_df.to_string(index=False))

    # Store flagged populations in uns for downstream steps
    strong_pops = pop_df.loc[pop_df['strong_spatial_purity'], 'population'].tolist()
    adata.uns['purity_spatial_umap_strong_populations'] = strong_pops
    print("\nFlagged Populations with strong spatial–Purity structure (not mirrored in UMAP):")
    print(strong_pops)
else:
    print("No populations met the minimum cell count threshold for correlation analysis.")

print("\n=== Top 20 Population × Sample_ID combinations by spatial Purity correlation strength (Spearman) ===")
ps_df = results_df[results_df['level'] == 'population_sample'].copy()
if not ps_df.empty:
    ps_df['max_abs_spatial_rho'] = ps_df[['spatial_x_spearman_rho', 'spatial_y_spearman_rho']].abs().max(axis=1)
    ps_df['max_abs_umap_rho'] = ps_df[['umap_x_spearman_rho', 'umap_y_spearman_rho']].abs().max(axis=1)
    summary_cols_ps = [
        'population', 'sample_id', 'n_cells',
        'spatial_x_spearman_rho', 'spatial_x_spearman_p',
        'spatial_y_spearman_rho', 'spatial_y_spearman_p',
        'umap_x_spearman_rho', 'umap_x_spearman_p',
        'umap_y_spearman_rho', 'umap_y_spearman_p',
        'max_abs_spatial_rho', 'max_abs_umap_rho',
    ]
    ps_df = ps_df[summary_cols_ps].sort_values('max_abs_spatial_rho', ascending=False).head(20)
    print(ps_df.to_string(index=False))
else:
    print("No Population × Sample_ID combinations met the minimum cell count threshold for correlation analysis.")

# Store full results in adata.uns for later steps
adata.uns['purity_spatial_umap_correlations'] = results_df
print("\nStored detailed correlation results in adata.uns['purity_spatial_umap_correlations'].")


ValueError: invalid literal for int() with base 10: '240010-R77_4C4'

### Agent Interpretation

Current analysis step failed to run. Try an alternative approach

## Next Steps
Step 1: Recompute spatial–Purity vs UMAP–Purity correlations without relying on external packages, identify well-powered Populations with strong spatial–Purity structure compared with UMAP, and store both the full correlation table and the list of flagged Populations in adata.uns.
Step 2: For each flagged Population (and, when n ≥ 200, Population×Sample_ID), assess QC confounding by computing Spearman correlations between Purity and UMI Count / Complexity and by comparing UMI Count and Complexity between high- and low-Purity strata (e.g., top vs bottom Purity quartile) using Wilcoxon rank-sum tests, explicitly classifying QC–Purity coupling as “modest” when |rho| < 0.3 and |log2 fold-change| < 0.5 with BH-FDR < 0.05, and summarize effect sizes and adjusted p-values.
Step 3: In Populations where Purity shows strong spatial structure but only modest QC confounding, perform within-Population subclustering using the precomputed neighbor graph/embeddings (restricting adata to that Population), assign Leiden subcluster labels, and test for differences in Purity distributions across subclusters via Kruskal–Wallis tests overall and within each Sample_ID where n ≥ 100.
Step 4: Within these selected Populations, run differential expression with sc.tl.rank_genes_groups (method='wilcoxon') comparing high- vs low-Purity cells and, separately, Purity-enriched vs Purity-depleted subclusters; from the most strongly DE genes construct per-Population Purity-up and Purity-down signatures, score them using sc.tl.score_genes, and model signature scores against Purity (with UMI Count, Complexity, and Sample_ID as covariates) using linear regression implemented via numpy/scipy, treating Purity as the main predictor and reporting regression coefficients and p-values in text.

## This code quantifies how strongly cell purity is associated with spatial and UMAP coordinates within each cell population and population–sample combination, to detect spatial purity structure that is not simply driven by the low‑dimensional embedding. It computes per-group Spearman (and Pearson) correlations between purity and each coordinate, applies FDR correction at the population level, flags populations with significant spatial–purity patterns not mirrored in UMAP, prints summaries, and stores all results in `adata.uns` for downstream analysis.

In [ ]:

import numpy as np
import pandas as pd
from scipy import stats

# Ensure required embeddings and columns exist
assert 'spatial' in adata.obsm, "Spatial coordinates not found in adata.obsm['spatial']"
assert 'X_umap' in adata.obsm, "X_umap embedding not found in adata.obsm['X_umap']"
for col in ['Populations', 'Sample_ID', 'Purity', 'UMI Count', 'Complexity']:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs")

# Attach coordinates to obs for convenience (idempotent)
spatial = adata.obsm['spatial']
umap = adata.obsm['X_umap']
adata.obs['spatial_x'] = spatial[:, 0]
adata.obs['spatial_y'] = spatial[:, 1]
adata.obs['umap_x'] = umap[:, 0]
adata.obs['umap_y'] = umap[:, 1]

# Ensure dtypes
adata.obs['Populations'] = adata.obs['Populations'].astype('category')
adata.obs['Sample_ID'] = adata.obs['Sample_ID'].astype('category')
adata.obs['Purity'] = pd.to_numeric(adata.obs['Purity'], errors='coerce')
adata.obs['UMI Count'] = pd.to_numeric(adata.obs['UMI Count'], errors='coerce')
adata.obs['Complexity'] = pd.to_numeric(adata.obs['Complexity'], errors='coerce')

min_cells_pop = 200
min_cells_pop_sample = 200

pop_col = 'Populations'
sample_col = 'Sample_ID'

# Helper: correlation summary using scipy.stats
def corr_summary(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    if x_valid.size < 3:
        return {
            'pearson_r': np.nan,
            'pearson_p': np.nan,
            'spearman_rho': np.nan,
            'spearman_p': np.nan,
            'n': int(x_valid.size),
        }
    pr, pp = stats.pearsonr(x_valid, y_valid)
    sr, sp = stats.spearmanr(x_valid, y_valid)
    return {
        'pearson_r': pr,
        'pearson_p': pp,
        'spearman_rho': sr,
        'spearman_p': sp,
        'n': int(x_valid.size),
    }

results = []

# Per-Population correlations
for pop, idx in adata.obs.groupby(pop_col).groups.items():
    # idx is already an Index of labels; use .loc directly
    sub = adata.obs.loc[idx]
    n_cells = sub.shape[0]
    if n_cells < min_cells_pop:
        continue
    purity = sub['Purity'].values

    csx = corr_summary(sub['spatial_x'].values, purity)
    csy = corr_summary(sub['spatial_y'].values, purity)
    cux = corr_summary(sub['umap_x'].values, purity)
    cuy = corr_summary(sub['umap_y'].values, purity)

    results.append({
        'level': 'population',
        'population': str(pop),
        'sample_id': 'ALL',
        'n_cells': n_cells,
        'spatial_x_spearman_rho': csx['spearman_rho'],
        'spatial_x_spearman_p': csx['spearman_p'],
        'spatial_y_spearman_rho': csy['spearman_rho'],
        'spatial_y_spearman_p': csy['spearman_p'],
        'umap_x_spearman_rho': cux['spearman_rho'],
        'umap_x_spearman_p': cux['spearman_p'],
        'umap_y_spearman_rho': cuy['spearman_rho'],
        'umap_y_spearman_p': cuy['spearman_p'],
    })

# Per-Population × Sample_ID correlations
for (pop, sample), idx in adata.obs.groupby([pop_col, sample_col]).groups.items():
    sub = adata.obs.loc[idx]
    n_cells = sub.shape[0]
    if n_cells < min_cells_pop_sample:
        continue
    purity = sub['Purity'].values

    csx = corr_summary(sub['spatial_x'].values, purity)
    csy = corr_summary(sub['spatial_y'].values, purity)
    cux = corr_summary(sub['umap_x'].values, purity)
    cuy = corr_summary(sub['umap_y'].values, purity)

    results.append({
        'level': 'population_sample',
        'population': str(pop),
        'sample_id': str(sample),
        'n_cells': n_cells,
        'spatial_x_spearman_rho': csx['spearman_rho'],
        'spatial_x_spearman_p': csx['spearman_p'],
        'spatial_y_spearman_rho': csy['spearman_rho'],
        'spatial_y_spearman_p': csy['spearman_p'],
        'umap_x_spearman_rho': cux['spearman_rho'],
        'umap_x_spearman_p': cux['spearman_p'],
        'umap_y_spearman_rho': cuy['spearman_rho'],
        'umap_y_spearman_p': cuy['spearman_p'],
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    print("No groups met the minimum cell-count thresholds; cannot proceed with spatial–Purity correlation analysis.")
else:
    # Population-level summaries
    pop_df = results_df[results_df['level'] == 'population'].copy()
    if pop_df.empty:
        print("No populations met the minimum cell-count threshold.")
        flagged_pops = []
    else:
        # Max |rho| across spatial x/y and UMAP x/y
        pop_df['max_abs_spatial_rho'] = pop_df[['spatial_x_spearman_rho', 'spatial_y_spearman_rho']].abs().max(axis=1)
        pop_df['max_abs_umap_rho'] = pop_df[['umap_x_spearman_rho', 'umap_y_spearman_rho']].abs().max(axis=1)

        # Minimum spatial p-value across x/y
        pop_df['min_spatial_p'] = pop_df[['spatial_x_spearman_p', 'spatial_y_spearman_p']].min(axis=1)

        # Manual Benjamini–Hochberg FDR (no external statsmodels)
        pvals = pop_df['min_spatial_p'].values
        m = len(pvals)
        order = np.argsort(pvals)
        ranked_p = pvals[order]
        bh_vals = ranked_p * m / (np.arange(m) + 1)
        # enforce monotonicity
        bh_vals = np.minimum.accumulate(bh_vals[::-1])[::-1]
        fdr = np.empty_like(bh_vals)
        fdr[order] = bh_vals
        pop_df['min_spatial_p_fdr'] = fdr

        # Flag strong spatial–Purity structure not mirrored in UMAP
        pop_df['strong_spatial_purity'] = (
            (pop_df['max_abs_spatial_rho'] - pop_df['max_abs_umap_rho'] >= 0.1) &
            (pop_df['min_spatial_p_fdr'] < 0.05)
        )
        flagged_pops = pop_df.loc[pop_df['strong_spatial_purity'], 'population'].tolist()

        print("\n=== Per-Population spatial vs UMAP Purity correlations (Spearman) ===")
        print(pop_df.sort_values('max_abs_spatial_rho', ascending=False).to_string(index=False))
        print("\nFlagged Populations with strong spatial–Purity structure (not mirrored in UMAP):")
        print(flagged_pops)

    # Population × Sample summaries (top 20 by spatial rho magnitude)
    ps_df = results_df[results_df['level'] == 'population_sample'].copy()
    if not ps_df.empty:
        ps_df['max_abs_spatial_rho'] = ps_df[['spatial_x_spearman_rho', 'spatial_y_spearman_rho']].abs().max(axis=1)
        ps_df['max_abs_umap_rho'] = ps_df[['umap_x_spearman_rho', 'umap_y_spearman_rho']].abs().max(axis=1)
        print("\n=== Top 20 Population × Sample_ID combinations by spatial Purity correlation strength (Spearman) ===")
        print(ps_df.sort_values('max_abs_spatial_rho', ascending=False).head(20).to_string(index=False))
    else:
        print("No Population × Sample_ID combinations met the minimum cell-count threshold.")

    # Store in adata.uns for downstream steps
    adata.uns['purity_spatial_umap_correlations'] = results_df
    adata.uns['purity_spatial_umap_strong_populations'] = flagged_pops
    print("\nStored detailed correlation results in adata.uns['purity_spatial_umap_correlations'] and flagged populations in adata.uns['purity_spatial_umap_strong_populations'].")



=== Per-Population spatial vs UMAP Purity correlations (Spearman) ===
     level population sample_id  n_cells  spatial_x_spearman_rho  spatial_x_spearman_p  spatial_y_spearman_rho  spatial_y_spearman_p  umap_x_spearman_rho  umap_x_spearman_p  umap_y_spearman_rho  umap_y_spearman_p  max_abs_spatial_rho  max_abs_umap_rho  min_spatial_p  min_spatial_p_fdr  strong_spatial_purity
population         PR       ALL     4673                0.578373          0.000000e+00                0.708686          0.000000e+00             0.559679       0.000000e+00             0.731440       0.000000e+00             0.708686          0.731440   0.000000e+00       0.000000e+00                  False
population         PT       ALL     3726                0.218452          1.711563e-41                0.500677         1.120799e-235             0.257546       1.599279e-57             0.499492      2.130189e-234             0.500677          0.499492  1.120799e-235      7.565393e-235                  False
po

### Agent Interpretation

The current step is doing exactly what you need for this hypothesis: it identifies populations where Purity has a strong spatial gradient that is not simply recapitulated in the low‑dimensional transcriptomic embedding. That’s the right substrate for asking whether Purity is more than a technical proxy.

Key observations from the output:

1. **You have several good candidate populations.**  
   The flagged populations with strong spatial–Purity structure but weaker UMAP–Purity coupling are:
   - **PD**
   - **PP**
   - **PS**
   - **PV**

   These are the ideal “test cases” for your hypothesis: Purity varies coherently across anatomy but isn’t just aligned with the main transcriptomic manifold.

2. **Effect sizes are in the “moderate but clear” regime.**  
   For flagged populations, the max |spatial rho| is ~0.19–0.39 at the population level:
   - PV: |spatial rho|max ≈ 0.39 vs |UMAP rho|max ≈ 0.20
   - PP: |spatial rho|max ≈ 0.26 vs |UMAP rho|max ≈ 0.13
   - PS: |spatial rho|max ≈ 0.22 vs |UMAP rho|max ≈ 0.11
   - PD: |spatial rho|max ≈ 0.19 vs |UMAP rho|max ≈ 0.07  

   These differences (~0.1–0.2) are not gigantic but are systematic and highly significant (FDR well below 0.05). They are large enough to be biologically meaningful while still being subtle enough that they plausibly reflect microenvironmental gradients or local contexts rather than wholesale cell-state differences.

3. **Population × Sample_ID results suggest sample‑specific structure.**  
   For example:
   - **PD, R78_4C12:** spatial_x rho ≈ 0.43, UMAP rho ≈ 0.08  
   - **PP, R78_4C12:** spatial_x rho ≈ 0.45, UMAP rho ≈ 0.21  
   - **PS, R78_4C12:** spatial_x rho ≈ −0.56 / +0.54 in x/y, UMAP rho almost 0  

   These combinations show very strong spatial–Purity gradients inside specific sections with only modest or discordant UMAP correlations, giving you well‑powered contexts for downstream QC and DE analyses.

Implications for your hypothesis:

- You now have concrete populations where **Purity behaves like a spatially structured microenvironmental variable, not just a generic quality metric**. The key remaining question is how tightly Purity tracks QC within these groups.
- If, in PD/PP/PS/PV (and their high‑n Population×Sample_ID subsets), Purity is only weakly correlated with UMI Count and Complexity and shows small QC shifts across Purity strata, that will strongly support the idea that Purity captures something biological and context‑specific.

Suggestions and cautions for the next steps:

1. **QC–Purity confounding analysis (next step in your plan):**
   - Focus first on **PD, PP, PS, PV** globally and then within their high‑n Population×Sample_ID combinations (e.g., PD–R78_4C12, PP–R78_4C12, PS–R78_4C12, any PV group ≥ 200 cells).
   - For each:
     - Compute Spearman correlations Purity vs UMI Count and Purity vs Complexity.
     - Define high‑ vs low‑Purity strata (top and bottom quartile is fine; consider requiring a minimum n per stratum).
     - Compare QC metrics between strata with Wilcoxon tests and compute **log2 fold‑changes**.
   - Apply your pre‑specified rule:
     - Classify QC–Purity coupling as **“modest”** when |rho| < 0.3 **and** |log2FC| < 0.5 with BH‑FDR < 0.05.
   - Summarize this per (Population, and Population×Sample_ID): a compact table with rho, log2FC, FDR, and the “modest/strong” label will be important for interpreting DE later.

   Interpretation guidance:
   - If most flagged groups fall into the “modest” bucket, that is **direct support** for the hypothesis that Purity is not primarily technical.
   - If one of the flagged populations shows strong QC coupling (e.g., |rho| > 0.4 or |log2FC| > 1), you may want to:
     - Either exclude it from downstream DE/subclustering focused on biology.
     - Or at least treat it as a “borderline / confounded” case and analyze more cautiously.

2. **Within‑Population subclustering (only where QC coupling is modest):**
   - Restrict to PD, PP, PS, PV **after** confirming modest QC–Purity relationships.
   - Use the precomputed neighbors/UMAP (avoiding refitting) on each subset and run Leiden to define subclusters.
   - You should:
     - Store Leiden labels in `.obs` per population (e.g., `Purity_subcluster_PD`).
     - Test Purity distributions across subclusters (Kruskal–Wallis overall, then within each Sample_ID with n ≥ 100).
   - What to look for:
     - Subclusters that differ in Purity but **do not show large QC differences** will be especially convincing evidence of microenvironmental structuring.
     - Patterns where high‑Purity subclusters map to specific spatial niches within the same population.

3. **Differential expression and Purity signatures:**
   - For each selected population (those with strong spatial–Purity and modest QC coupling), run two DE analyses:
     1. **High vs low Purity strata.**
     2. **Purity‑enriched vs Purity‑depleted subclusters.**
   - From overlapping DE genes, define **Purity‑up** and **Purity‑down** signatures per population.
   - Score them with `sc.tl.score_genes` and then do your planned linear regression:
     - Outcome: signature score  
     - Predictors: Purity (main), UMI Count, Complexity, and Sample_ID (encoded via dummy variables).
   - Key readouts:
     - The regression coefficient of Purity and its p‑value after accounting for QC and Sample_ID.
     - Populations where Purity remains a strong predictor after adjustment are the cleanest validations of microenvironment‑linked biology.

4. **Keep spatial specificity in mind:**
   - Several non‑flagged populations (e.g., PR, PT, PN) have **very strong** spatial and UMAP Purity correlations (|rho| ~0.5–0.7) that are roughly aligned.  
     - These might be dominated by large‑scale anatomical gradients or broad cell‑state differences; they are less ideal for your current hypothesis, but later you could contrast them to PD/PP/PS/PV as “UMAP‑aligned” controls.
   - Within the flagged populations, leverage the Population×Sample_ID breakdown:
     - Some sections may show particularly sharp spatial Purity patterns (e.g., PD–R78_4C12), which are natural targets for spatial visualization and interpretation once you know the DE programs.

5. **Practical code feedback:**
   - The correlation/BH implementation is sound and self‑contained, satisfying the “no external package” constraint.
   - For downstream steps, re‑use the same patterns:
     - Implement BH by hand for the QC–Purity Wilcoxon tests and for DE effect‑size summaries.
   - Consider storing a more processed summary in `adata.uns` (e.g., `purity_spatial_umap_summary`) that includes `max_abs_spatial_rho`, `max_abs_umap_rho`, `min_spatial_p_fdr`, and `strong_spatial_purity` for easier downstream filtering.

In summary, this step has successfully identified a short list of populations (PD, PP, PS, PV) where Purity is spatially structured but not simply a UMAP axis. The next QC‑confounding and within‑population analyses will directly test your hypothesis by checking whether, in these populations, Purity retains biological structure after QC is accounted for.

## Next Steps
Step 1: Quantify QC–Purity coupling within each flagged spatial–Purity population (PD, PP, PS, PV) and its well-powered Population×Sample_ID strata (n ≥ 200) by computing Spearman correlations between Purity and UMI Count / Complexity and by contrasting UMI Count and Complexity between high- and low-Purity strata (top vs bottom Purity quartile) using Wilcoxon rank-sum tests, applying BH-FDR correction within each level×population×qc_metric family and classifying each stratum as having modest vs strong QC–Purity coupling based on |rho| and |log2 fold-change| thresholds, while also recording an FDR-based significance flag.
Step 2: Restrict downstream analyses to populations and Population×Sample_ID strata that both show strong spatial–Purity structure and predominantly modest QC–Purity coupling, then perform within-population Leiden subclustering on the precomputed neighborhood graph/embedding (subsetting adata to each such population in turn), annotate subcluster labels back to the full AnnData, and test for differences in Purity and QC metric distributions across subclusters overall and within each Sample_ID (n ≥ 100) via Kruskal–Wallis tests, explicitly flagging subclusters that are Purity-enriched or -depleted while confirming that QC metrics remain balanced.
Step 3: Within these QC-robust, spatially structured populations and subclusters, conduct differential expression analyses using sc.tl.rank_genes_groups (method='wilcoxon') comparing high- vs low-Purity cells and, separately, Purity-enriched vs Purity-depleted subclusters, derive concise Purity-up and Purity-down gene signatures per population from strongly DE genes (FDR- and log2FC-filtered), score each cell with sc.tl.score_genes, and fit per-population linear regression models (via numpy/scipy) that relate signature scores to Purity while adjusting for UMI Count, Complexity, and Sample_ID, reporting regression coefficients, p-values, and goodness-of-fit metrics to quantify the extent to which Purity-associated programs persist after QC adjustment.

## This code quantifies QC–Purity coupling within spatially structured cardiac populations and their well-powered Population×Sample_ID strata by computing Spearman correlations and Wilcoxon tests between Purity and QC metrics, applies BH-FDR correction within each level×population×qc_metric family, classifies effect-size-based modest vs strong coupling, adds an explicit FDR-based significance flag, and stores/prints the results for downstream filtering.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Retrieve flagged populations with strong spatial–Purity structure
strong_pops = adata.uns.get('purity_spatial_umap_strong_populations', [])
if not strong_pops:
    print("No populations with strong spatial–Purity structure found in adata.uns['purity_spatial_umap_strong_populations'].")

# Ensure required columns exist and are numeric
required_cols = ['Populations', 'Sample_ID', 'Purity', 'UMI Count', 'Complexity']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs")

for col in ['Purity', 'UMI Count', 'Complexity']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

adata.obs['Populations'] = adata.obs['Populations'].astype('category')
adata.obs['Sample_ID'] = adata.obs['Sample_ID'].astype('category')

min_cells_group = 200

# Helper: Benjamini–Hochberg FDR (self-contained)
def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    if m == 0:
        return np.array([])
    order = np.argsort(pvals)
    ranked = pvals[order]
    bh = ranked * m / (np.arange(m) + 1)
    bh = np.minimum.accumulate(bh[::-1])[::-1]
    out = np.empty_like(bh)
    out[order] = bh
    return out

# Helper: compute Spearman rho and p
def spearman_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    rho, p = stats.spearmanr(x[mask], y[mask])
    return rho, p

# Container for results
records = []

# Analyze each flagged population
for pop in strong_pops:
    pop_mask = adata.obs['Populations'] == pop
    if pop_mask.sum() < min_cells_group:
        continue

    # Global (population-level) analysis
    sub = adata.obs.loc[pop_mask, ['Purity', 'UMI Count', 'Complexity', 'Sample_ID']].copy()
    n_cells = sub.shape[0]

    # Define high/low Purity strata (top/bottom quartile)
    purity_vals = sub['Purity'].values
    q25, q75 = np.percentile(purity_vals, [25, 75])
    high_mask = purity_vals >= q75
    low_mask = purity_vals <= q25

    # Ensure both strata have enough cells (also require at least 10% of cells in each stratum)
    if (high_mask.sum() < 20 or low_mask.sum() < 20 or
        high_mask.sum() < 0.1 * n_cells or low_mask.sum() < 0.1 * n_cells):
        print(f"Population {pop}: insufficient or highly imbalanced high/low Purity strata at population level, skipping.")
    else:
        for qc_col in ['UMI Count', 'Complexity']:
            x = sub[qc_col].values
            # Spearman correlations
            rho, p_rho = spearman_corr(purity_vals, x)

            # Wilcoxon rank-sum between high and low Purity
            stat_w, p_w = stats.ranksums(x[high_mask], x[low_mask])
            # Effect size as log2 fold-change (high / low)
            mean_high = np.mean(x[high_mask])
            mean_low = np.mean(x[low_mask])
            if mean_low > 0:
                log2fc = np.log2((mean_high + 1e-8) / (mean_low + 1e-8))
            else:
                log2fc = np.nan
                print(f"Warning: mean {qc_col} for low-Purity stratum <= 0 in population {pop} (population level); setting log2FC to NaN.")

            records.append({
                'level': 'population',
                'population': pop,
                'sample_id': 'ALL',
                'qc_metric': qc_col,
                'n_cells': n_cells,
                'n_high': int(high_mask.sum()),
                'n_low': int(low_mask.sum()),
                'spearman_rho': rho,
                'spearman_p': p_rho,
                'wilcoxon_stat': stat_w,
                'wilcoxon_p': p_w,
                'mean_high': mean_high,
                'mean_low': mean_low,
                'log2fc_high_vs_low': log2fc,
            })

    # Per-Population×Sample_ID analysis
    for sample, idx in sub.groupby('Sample_ID').groups.items():
        sample_mask = pop_mask & (adata.obs['Sample_ID'] == sample)
        if sample_mask.sum() < min_cells_group:
            continue
        sub_ps = adata.obs.loc[sample_mask, ['Purity', 'UMI Count', 'Complexity']].copy()
        n_cells_ps = sub_ps.shape[0]
        purity_vals_ps = sub_ps['Purity'].values
        q25_ps, q75_ps = np.percentile(purity_vals_ps, [25, 75])
        high_mask_ps = purity_vals_ps >= q75_ps
        low_mask_ps = purity_vals_ps <= q25_ps

        # Require minimum absolute and relative size per stratum
        if (high_mask_ps.sum() < 15 or low_mask_ps.sum() < 15 or
            high_mask_ps.sum() < 0.1 * n_cells_ps or low_mask_ps.sum() < 0.1 * n_cells_ps):
            continue

        for qc_col in ['UMI Count', 'Complexity']:
            x_ps = sub_ps[qc_col].values
            rho_ps, p_rho_ps = spearman_corr(purity_vals_ps, x_ps)
            stat_w_ps, p_w_ps = stats.ranksums(x_ps[high_mask_ps], x_ps[low_mask_ps])
            mean_high_ps = np.mean(x_ps[high_mask_ps])
            mean_low_ps = np.mean(x_ps[low_mask_ps])
            if mean_low_ps > 0:
                log2fc_ps = np.log2((mean_high_ps + 1e-8) / (mean_low_ps + 1e-8))
            else:
                log2fc_ps = np.nan
                print(f"Warning: mean {qc_col} for low-Purity stratum <= 0 in population {pop}, sample {sample}; setting log2FC to NaN.")

            records.append({
                'level': 'population_sample',
                'population': pop,
                'sample_id': str(sample),
                'qc_metric': qc_col,
                'n_cells': n_cells_ps,
                'n_high': int(high_mask_ps.sum()),
                'n_low': int(low_mask_ps.sum()),
                'spearman_rho': rho_ps,
                'spearman_p': p_rho_ps,
                'wilcoxon_stat': stat_w_ps,
                'wilcoxon_p': p_w_ps,
                'mean_high': mean_high_ps,
                'mean_low': mean_low_ps,
                'log2fc_high_vs_low': log2fc_ps,
            })

# Assemble results into a DataFrame
qc_df = pd.DataFrame.from_records(records)

if qc_df.empty:
    print("No QC–Purity results were computed (possibly due to lack of flagged populations or insufficient cell counts).")
else:
    # Apply BH-FDR correction separately for: level × population × qc_metric
    fdr_vals = np.empty(qc_df.shape[0], dtype=float)
    # Also record significance based on FDR < 0.05
    signif_flag = np.zeros(qc_df.shape[0], dtype=bool)

    for (lvl, pop, qc), sub_df in qc_df.groupby(['level', 'population', 'qc_metric']):
        idx = sub_df.index.values
        pvals = sub_df['wilcoxon_p'].values
        fdr = bh_fdr(pvals)
        fdr_vals[idx] = fdr
        signif_flag[idx] = fdr < 0.05

    qc_df['wilcoxon_p_fdr'] = fdr_vals
    qc_df['qc_purity_significant'] = signif_flag

    # Classify QC–Purity coupling as modest vs strong based purely on effect sizes
    # Criteria: "modest" when |rho| < 0.3 AND |log2FC| < 0.5 (regardless of significance);
    # else "strong_or_moderate".
    qc_df['qc_purity_coupling'] = np.where(
        (qc_df['spearman_rho'].abs() < 0.3) & (qc_df['log2fc_high_vs_low'].abs() < 0.5),
        'modest',
        'strong_or_moderate'
    )

    # Store in adata.uns
    adata.uns['purity_qc_coupling'] = qc_df

    # Print concise summaries per population
    print("\n=== QC–Purity coupling summary for flagged spatial–Purity populations ===")
    summary_cols = [
        'level', 'population', 'sample_id', 'qc_metric', 'n_cells', 'n_high', 'n_low',
        'spearman_rho', 'spearman_p', 'wilcoxon_p', 'wilcoxon_p_fdr',
        'log2fc_high_vs_low', 'qc_purity_coupling', 'qc_purity_significant'
    ]
    # Sort by population and QC metric
    print(qc_df[summary_cols].sort_values(['population', 'level', 'qc_metric', 'sample_id']).to_string(index=False))

    # Also report which (population, qc_metric) pairs are predominantly modest vs strong
    agg = (
        qc_df
        .groupby(['population', 'qc_metric'])['qc_purity_coupling']
        .value_counts()
        .unstack(fill_value=0)
    )
    print("\n=== Counts of modest vs strong_or_moderate QC–Purity coupling per population and QC metric ===")
    print(agg.to_string())


=== QC–Purity coupling summary for flagged spatial–Purity populations ===
            level population sample_id  qc_metric  n_cells  n_high  n_low  spearman_rho    spearman_p    wilcoxon_p  wilcoxon_p_fdr  log2fc_high_vs_low qc_purity_coupling  qc_purity_significant
       population         PD       ALL Complexity    16624    4156   4156     -0.238434 1.416375e-213 1.772420e-166   1.772420e-166           -0.168011             modest                   True
       population         PD       ALL  UMI Count    16624    4156   4156     -0.038099  8.941144e-07  9.438446e-07    9.438446e-07           -0.089950             modest                   True
population_sample         PD   R77_4C4 Complexity     4817    1205   1205     -0.151023  5.679905e-26  3.761016e-27    3.761016e-27           -0.115607             modest                   True
population_sample         PD  R78_4C12 Complexity     5417    1355   1355     -0.218859  9.488624e-60  1.061925e-44    1.592887e-44           -0.1429

### Agent Interpretation

These results generally support the hypothesis that Purity is not simply a QC surrogate, but the strength of QC–Purity coupling varies by population and metric:

1. **Overall pattern across PD, PP, PS, PV**
   - All four populations show **many “modest” QC–Purity couplings**, especially for UMI Count:
     - UMI Count: entirely “modest” in PD, PP, PS, PV (across population and population×sample strata).
     - Complexity: “modest” in PD; mixed in PP; clearly **strong** in PS and PV.
   - Because your criteria for “modest” are effect-size based (|rho|<0.3 & |log2FC|<0.5), the highly significant p-values simply reflect large n and don’t contradict modest coupling.

2. **Population-specific conclusions for downstream analyses**

   - **PD**
     - Complexity: all 4 strata (1 pop-level + 3 sample-level) are **modest** (rho ≈ −0.15 to −0.30; log2FC ≈ −0.11 to −0.21).
     - UMI Count: all 4 strata **modest** (rho small, |log2FC| < 0.2).
     - PD is a **prime QC-robust candidate**: strong spatial–Purity structure (by construction) and consistently modest QC–Purity coupling in both metrics.
     - PD should be prioritized for the next steps (Leiden subclustering and Purity-associated DE/regression) with high confidence that Purity captures biology beyond QC.

   - **PP**
     - Complexity:
       - Population-level and two samples (R78_4C12, R78_4C15) are flagged **strong_or_moderate** (rho ~ −0.31 to −0.37, log2FC ~ −0.19 to −0.24).
       - Only R77_4C4 is “modest”.
     - UMI Count:
       - All strata “modest”; population-level rho ~ 0.009 and log2FC ~ −0.02.
     - Interpretation:
       - In PP, **Purity is somewhat entangled with Complexity**, but **not with UMI Count**.
       - For downstream analysis, PP is usable, but with **caution**:
         - Prefer to use **UMI Count and Complexity as covariates** later, and be prepared that some Purity–biology associations might be partially confounded by Complexity.
         - In sample R77_4C4 (Complexity modest), PP is especially clean; you could consider **sensitivity analyses restricted to R77_4C4** or to subclusters where Complexity distributions remain balanced.

   - **PS**
     - Complexity:
       - All strata (pop-level + 3 samples) are **strong_or_moderate** with **large negative effects**:
         - rho ~ −0.43 to −0.50; log2FC ~ −0.56 to −0.75.
       - This is **substantial**: high-Purity cells have markedly lower Complexity.
     - UMI Count:
       - All strata “modest” with small positive rho and small log2FC (~0.10–0.18).
     - Interpretation:
       - In PS, Purity is **strongly anti-correlated with Complexity**, even though UMI Count coupling is modest.
       - Purity here is likely capturing a combination of biology and technical diversity in library complexity.
       - For your hypothesis (Purity not just QC), PS is **problematic**: Complexity is too tightly linked to Purity to call the QC–Purity coupling “modest.”
       - I would either:
         - **Exclude PS from the main “QC-robust” set**, or
         - Include PS only with **strict downstream controls**, e.g.:
           - Very stringent matching of Complexity across Purity strata in subclusters, or
           - Heavy use of Complexity as a covariate and **sensitivity checks** (e.g., regression residual analyses where Complexity has minimal residual association with Purity).

   - **PV**
     - Complexity:
       - All strata “strong_or_moderate” with strong negative correlations:
         - pop-level rho = −0.49, log2FC ~ −0.41; per-sample rho ~ −0.37 to −0.42, log2FC ~ −0.27 to −0.35.
     - UMI Count:
       - All strata “modest” with small positive rho (~0.00–0.16) and moderate log2FC in the population-level high vs low (~0.37).
     - Interpretation:
       - Very similar to PS: **Purity is tightly coupled to Complexity** but only modestly to UMI Count.
       - Again, the strong Complexity coupling compromises the hypothesis that Purity is “mostly independent” of QC in this population.
       - Recommendation similar to PS: **do not treat PV as QC-robust** without explicit strategies to account for the Complexity conflation.

3. **Implications for the hypothesis**

   - The results **partially validate** your hypothesis:
     - There are clearly populations (notably PD, and to a lesser extent PP when focusing on UMI Count) where Purity’s relationship with QC is modest by your effect-size criteria.
     - This is especially compelling because I see **significant but small effects**—typical of truly weak technical confounding in large datasets.
   - However, for **PS and PV**, Purity is very strongly coupled to Complexity. In those contexts, it is difficult to argue that Purity is “not merely a QC surrogate,” at least with respect to Complexity.

4. **How to refine the next steps**

   a. **Selection of populations / strata for downstream analyses**
   - When you “restrict downstream analyses to populations and Population×Sample_ID strata that… show predominantly modest QC–Purity coupling”:
     - **Clearly include:**
       - PD (all levels, both QC metrics).
       - PP for UMI Count; for Complexity, possibly only R77_4C4.
     - **Potentially include with caveats / sensitivity analyses:**
       - PP strata where Complexity is strong_or_moderate (R78_4C12, R78_4C15).
     - **Exclude or treat as separate, QC-confounded sets:**
       - PS and PV, at least if Complexity is a major part of your QC definition.
   - You might explicitly encode a mask in `adata.obs` or `adata.uns` capturing **QC-robust cells** (e.g., PD + PP + maybe specific PP×Sample_ID subsets), and run subclustering/DE only within this mask for the main analysis.

   b. **Subclustering and balance checks (next planned step)**
   - For PD and the QC-robust PP strata, proceed with:
     - Leiden subclustering on precomputed neighbors/embedding.
     - Kruskal–Wallis tests for differences in **Purity**, **UMI Count**, and **Complexity** across subclusters and per-sample.
   - When interpreting subclusters as “Purity-enriched/-depleted,” pay particular attention to:
     - Whether Complexity remains balanced in those subclusters, especially in PP, PS, PV.
     - If you find subclusters where both Purity and Complexity differ strongly, mark those as **likely QC-confounded** and:
       - Either drop them from downstream DE, or
       - Treat them as “QC-sensitive” in separate sensitivity analyses.

   c. **Design of regression models (step 3)**
   - The current results reinforce the need for **multivariable regression**:
     - For PD and PP, you can be confident that after including UMI Count & Complexity as covariates, residual Purity effects will approximate “biological” variation.
     - For PS and PV, even if you include Complexity as a covariate, the very strong coupling raises collinearity issues:
       - You may see inflated standard errors or unstable coefficients for Purity vs Complexity.
       - Consider checking **variance inflation factors** (VIF) or pairwise correlations among covariates in those populations.
   - You may want to run the regression step **twice**:
     1. In the QC-robust populations/strata (PD + PP), to define your core Purity signatures.
     2. Optionally in PS/PV, but treat these results as **exploratory** and see whether Purity-associated programs overlap with the QC-robust signatures or are dominated by library-composition-like patterns.

5. **Additional small technical notes / options for iteration**

   - Your thresholds (|rho| < 0.3, |log2FC| < 0.5) are sensible but slightly permissive. For example, in PP complexity, rho ≈ −0.31 (just over the line) plus modest log2FC already triggers “strong_or_moderate.”
     - If you want a **stricter QC-robust core**, you could tighten these cutoffs for the principal analysis (e.g., |rho| < 0.25 & |log2FC| < 0.4), then see which populations persist.
   - Within PS and PV, you might explore whether **within narrow Complexity bands** (e.g., stratify or residualize Complexity), Purity still shows spatial structure. If so, that would support biological meaning even in QC-confounded contexts.

In summary, these QC–Purity coupling results strongly justify moving forward with PD (and largely PP) as **QC-robust, spatially structured populations** for the next subclustering and DE steps. PS and PV likely require harsher treatment (exclusion or heavy QC-adjustment and sensitivity analyses) to uphold the hypothesis that Purity reflects microenvironment beyond technical quality.

## Next Steps
Step 1: Restrict to QC-robust, spatially structured populations by selecting PD and PP (previously identified as having strong spatial–Purity structure with predominantly modest QC–Purity coupling), perform within-population Leiden subclustering using the existing neighbors/embedding (subsetting adata to each population in turn), write back population-specific subcluster labels into adata.obs, and for each population and each Sample_ID with n ≥ 100 per population, use Kruskal–Wallis tests to quantify whether Purity, UMI Count, and Complexity differ across subclusters, explicitly flagging subclusters that are Purity-enriched or -depleted while maintaining modest QC differences.
Step 2: Within PD and PP, and focusing on subclusters that show Purity shifts but balanced QC metrics, perform differential expression analysis using sc.tl.rank_genes_groups (method='wilcoxon') comparing (i) high- vs low-Purity cells (top vs bottom Purity quartile) and (ii) Purity-enriched vs Purity-depleted subclusters, derive concise Purity-up and Purity-down signatures per population from strongly DE genes (FDR- and log2FC-filtered), score each cell with sc.tl.score_genes, and fit per-population linear regression models (via numpy/scipy) that relate signature scores to Purity while adjusting for UMI Count, Complexity, and Sample_ID, reporting regression coefficients and p-values to quantify how strongly Purity-associated programs persist after QC adjustment.

## This code subclusters previously defined, QC-robust populations (PD, PP) using Leiden clustering and then statistically tests whether cell purity and sequencing QC metrics differ across these subclusters, both globally and within each sample. It then flags subclusters with significantly enriched or depleted purity but relatively stable QC metrics to identify biologically meaningful purity-associated states for downstream differential expression/signature analyses.

In [ ]:

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats

# We focus on QC-robust, spatially structured populations identified previously
qc_robust_pops = ['PD', 'PP']

# Ensure required columns are present
for col in ['Populations', 'Sample_ID', 'Purity', 'UMI Count', 'Complexity', 'leiden']:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs")

# Enforce appropriate dtypes and numeric QC metrics for robustness
adata.obs['Populations'] = adata.obs['Populations'].astype('category')
adata.obs['Sample_ID'] = adata.obs['Sample_ID'].astype('category')
for col in ['Purity', 'UMI Count', 'Complexity']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

# Use existing neighbors/embedding; we assume they were computed globally already

min_cells_pop = 500  # ensure enough cells for stable subclustering
min_cells_sample_for_kw = 100  # per-population per-sample threshold for KW tests

subcluster_results = []

for pop in qc_robust_pops:
    pop_mask = adata.obs['Populations'] == pop
    n_pop = int(pop_mask.sum())
    if n_pop < min_cells_pop:
        print(f"Population {pop}: only {n_pop} cells, below subclustering threshold, skipping.")
        continue

    print(f"\n=== Subclustering population {pop} (n={n_pop}) ===")
    adata_pop = adata[pop_mask].copy()

    # Run Leiden clustering within this population using precomputed neighbors/embedding if available
    # Prefer neighbors based on the same representation as global, but recompute locally for robustness
    if 'X_pca' in adata_pop.obsm_keys():
        sc.pp.neighbors(adata_pop, use_rep='X_pca')
    else:
        sc.pp.neighbors(adata_pop)

    sc.tl.leiden(adata_pop, resolution=1.0, key_added=f'leiden_{pop}_sub')

    # Map subcluster labels back to the main AnnData (avoid categorical dtype mismatch)
    col_name = f'leiden_{pop}_sub'
    # always ensure this column exists and is non-categorical before assignment
    adata.obs[col_name] = pd.Series(index=adata.obs.index, dtype='object')
    adata.obs.loc[pop_mask, col_name] = adata_pop.obs[col_name].astype(str).values
    # then convert to category after assignment
    adata.obs[col_name] = adata.obs[col_name].astype('category')

    # For each population, test whether Purity and QC metrics differ across subclusters (global and per-sample)
    obs_pop = adata.obs.loc[pop_mask, ['Purity', 'UMI Count', 'Complexity', 'Sample_ID', col_name]].copy()

    # Helper for Kruskal–Wallis across subclusters
    def kw_by_subcluster(df, value_col, group_col):
        grouped = df.groupby(group_col)[value_col]
        groups = [v.dropna().values for _, v in grouped]
        labels = list(grouped.groups.keys())
        if len(groups) < 2:
            return np.nan, np.nan, labels
        # Require at least some minimum n per group
        if any(len(g) < 10 for g in groups):
            return np.nan, np.nan, labels
        stat, p = stats.kruskal(*groups)
        return stat, p, labels

    # Global (within-population) KW tests
    for metric in ['Purity', 'UMI Count', 'Complexity']:
        stat, p, sub_labels_list = kw_by_subcluster(obs_pop, metric, col_name)
        subcluster_results.append({
            'population': pop,
            'sample_id': 'ALL',
            'metric': metric,
            'level': 'population',
            'n_cells': int(obs_pop.shape[0]),
            'n_subclusters': len(sub_labels_list),
            'kw_stat': stat,
            'kw_p': p,
        })
        print(f"Population {pop}, ALL samples, {metric}: KW stat={stat:.3f}, p={p:.3e}, n_subclusters={len(sub_labels_list)}")

    # Per-sample KW tests within each population
    for sample_id, df_sample in obs_pop.groupby('Sample_ID'):
        n_sample = df_sample.shape[0]
        if n_sample < min_cells_sample_for_kw:
            continue
        for metric in ['Purity', 'UMI Count', 'Complexity']:
            stat, p, sub_labels_list = kw_by_subcluster(df_sample, metric, col_name)
            subcluster_results.append({
                'population': pop,
                'sample_id': str(sample_id),
                'metric': metric,
                'level': 'population_sample',
                'n_cells': int(n_sample),
                'n_subclusters': len(sub_labels_list),
                'kw_stat': stat,
                'kw_p': p,
            })
            print(f"Population {pop}, Sample {sample_id}, {metric}: KW stat={stat:.3f}, p={p:.3e}, n_subclusters={len(sub_labels_list)}")

    # Identify Purity-enriched/depleted subclusters within each population
    sub_means = (
        obs_pop
        .groupby(col_name)[['Purity', 'UMI Count', 'Complexity']]
        .agg(['mean', 'std', 'count'])
    )
    print(f"\nSummary of Purity and QC metrics by subcluster in population {pop}:")
    print(sub_means)

    # Flag subclusters with Purity clearly above/below population mean while keeping QC differences modest
    pop_mean_purity = obs_pop['Purity'].mean()
    pop_sd_purity = obs_pop['Purity'].std()
    pop_mean_umi = obs_pop['UMI Count'].mean()
    pop_mean_comp = obs_pop['Complexity'].mean()

    enriched = []
    depleted = []
    for cl in obs_pop[col_name].cat.categories:
        cl_mask = obs_pop[col_name] == cl
        n_cl = int(cl_mask.sum())
        if n_cl < 50:
            continue
        cl_purity_mean = obs_pop.loc[cl_mask, 'Purity'].mean()
        cl_umi_mean = obs_pop.loc[cl_mask, 'UMI Count'].mean()
        cl_comp_mean = obs_pop.loc[cl_mask, 'Complexity'].mean()

        umi_shift = abs(cl_umi_mean - pop_mean_umi) / (pop_mean_umi + 1e-8)
        comp_shift = abs(cl_comp_mean - pop_mean_comp) / (pop_mean_comp + 1e-8)

        # Purity-enriched if > 0.5 SD above mean and QC shifts modest (|delta|/mean < 0.25)
        if cl_purity_mean >= pop_mean_purity + 0.5 * pop_sd_purity and umi_shift < 0.25 and comp_shift < 0.25:
            enriched.append(cl)
        # Purity-depleted if < 0.5 SD below mean with modest QC shifts
        if cl_purity_mean <= pop_mean_purity - 0.5 * pop_sd_purity and umi_shift < 0.25 and comp_shift < 0.25:
            depleted.append(cl)

    print(f"Purity-enriched subclusters in {pop} with modest QC shifts: {enriched}")
    print(f"Purity-depleted subclusters in {pop} with modest QC shifts: {depleted}")

    # Store flags in adata.uns for downstream DE/signature analysis
    adata.uns.setdefault('purity_subclusters', {})[pop] = {
        'enriched': enriched,
        'depleted': depleted,
    }

# Collect and store KW results
if subcluster_results:
    subcluster_results_df = pd.DataFrame(subcluster_results)
    adata.uns['purity_subcluster_kw_tests'] = subcluster_results_df
    print("\nStored Kruskal–Wallis test results in adata.uns['purity_subcluster_kw_tests'].")
    print("Kruskal–Wallis summary (first 20 rows):")
    print(subcluster_results_df.head(20).to_string(index=False))
else:
    print("No subclustering/KW results were generated (possibly due to insufficient cell counts).")



=== Subclustering population PD (n=16624) ===
computing neighbors


computing PCA


    with n_comps=50


    finished (0:00:00)


    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:02)


...

    finished (0:00:00)


    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)


running Leiden clustering


    finished: found 10 clusters and added
    'leiden_PP_sub', the cluster labels (adata.obs, categorical) (0:00:00)


Population PP, ALL samples, Purity: KW stat=163.851, p=1.176e-30, n_subclusters=10
Population PP, ALL samples, UMI Count: KW stat=664.634, p=2.760e-137, n_subclusters=10
Population PP, ALL samples, Complexity: KW stat=190.321, p=3.527e-36, n_subclusters=10
Population PP, Sample R77_4C4, Purity: KW stat=34.325, p=7.834e-05, n_subclusters=10
Population PP, Sample R77_4C4, UMI Count: KW stat=159.234, p=1.071e-29, n_subclusters=10
Population PP, Sample R77_4C4, Complexity: KW stat=74.672, p=1.834e-12, n_subclusters=10
Population PP, Sample R78_4C12, Purity: KW stat=144.366, p=1.293e-26, n_subclusters=10
Population PP, Sample R78_4C12, UMI Count: KW stat=291.417, p=1.724e-57, n_subclusters=10
Population PP, Sample R78_4C12, Complexity: KW stat=76.472, p=8.087e-13, n_subclusters=10
Population PP, Sample R78_4C15, Purity: KW stat=38.194, p=1.609e-05, n_subclusters=10
Population PP, Sample R78_4C15, UMI Count: KW stat=199.655, p=3.913e-38, n_subclusters=10
Population PP, Sample R78_4C15, Compl

### Agent Interpretation

The current step shows that PD and PP do have transcriptionally distinct subclusters with highly significant differences in Purity, but these Purity differences are tightly entangled with QC (UMI Count and Complexity). Under the strict criterion you imposed (|ΔQC|/mean < 0.25 and ≥0.5 SD shift in Purity), no subcluster qualifies as Purity-enriched or -depleted with “modest” QC shifts in either population.

Key points from the results:

- Strong substructure exists:
  - PD: 8 Leiden subclusters (n=16,624 cells).
  - PP: 10 Leiden subclusters (n=5,458 cells).
- Very strong Kruskal–Wallis signals for Purity, UMI Count, and Complexity across subclusters:
  - PD population-wide Purity: KW p ~ 10⁻¹⁴⁷; UMI and Complexity also extremely significant.
  - PP similar pattern: robust differences for all three metrics.
- Mean Purity differences between subclusters are modest in absolute terms (∼0.38–0.45), but still statistically robust.
- Crucially, subclusters that deviate in Purity almost always show noticeable shifts in UMI Count and/or Complexity—your current thresholds find zero subclusters where Purity shifts “cleanly” without QC shifts. This holds for both PD and PP.

Implications for the hypothesis

The strictest reading of your hypothesis is not yet supported:

> “Within PD and PP, there exist transcriptionally distinct subclusters whose Purity differences cannot be explained by UMI Count or Complexity…”

At the subcluster level, Purity and QC are still significantly associated: every Leiden cluster with a strong Purity deviation also tends to show non-trivial QC shifts. The KW results show that “cluster identity” explains variance in both Purity and QC, and your rule-based screen fails to find any clusters where Purity is clearly shifted but QC is “flat” by your definition.

However:

- The effect sizes in Purity are relatively small (e.g., PD mean Purity ~0.38–0.45; PP ~0.38–0.45), whereas UMI differences can be larger. It’s still possible that, at the cell level, Purity is partially independent of QC after adjustment.
- Your thresholds (0.5 SD in Purity, <25% relative QC shift, n ≥ 50) are quite stringent; relaxing them may reveal “good enough” clusters that are biologically interpretable, even if not perfectly decoupled from QC.
- Cluster-level averages may obscure substructure: cells within the same subcluster may still display Purity variation that is only weakly explained by QC.

So, this step suggests that technical variation is still strongly coupled to the Leiden-defined transcriptional structure, making it difficult to find “pure” microenvironmental states at the subcluster average level. But it does not rule out Purity capturing a meaningful microenvironmental gradient within and across these clusters.

Concrete suggestions for next steps

1. **Relax and refine the Purity/QC subcluster screen (still at this step)**

   Before moving to DE:

   - Consider milder thresholds to define candidate Purity-enriched/depleted subclusters:
     - e.g., |ΔPurity| ≥ 0.3 SD from the population mean rather than 0.5 SD, and/or allow QC shifts up to 0.35–0.4 relative instead of 0.25.
   - Compute **standardized effect sizes** per cluster:
     - z_Purity = (mean_cl – mean_pop) / sd_pop  
     - z_UMI, z_Complexity similarly.
     - Look for clusters where |z_Purity| is large relative to |z_UMI| and |z_Complexity| (e.g., |z_Purity| – max(|z_UMI|, |z_Complexity|) > 0.5).
   - Alternatively, regress Purity on UMI and Complexity within each population and look at **cluster-level residual means**; identify clusters enriched for positive vs negative residuals. That directly defines “Purity beyond QC”.

   These approaches will give you a graded ranking of subclusters by “Purity specificity” instead of a hard pass/fail list, which will be useful for downstream DE.

2. **Use cell-level models rather than cluster means to decode Purity–QC relationships**

   To address the core hypothesis, move beyond KW tests and cluster means:

   - Within PD and PP, fit per-population linear or generalized additive models:
     - Purity ~ β₀ + β₁ * log10(UMI Count) + β₂ * Complexity + batch (Sample_ID) + ε.
   - Examine **R²**: how much Purity variance is explained by QC?
   - Compute per-cell **Purity residuals** from this model; these residuals are, by construction, orthogonal to QC.
   - Redefine “high-Purity” vs “low-Purity” groups using **residual Purity** (e.g., top vs bottom quartile of residuals) when you move to DE. That directly tests whether there is expression structure associated with Purity beyond QC.

   This will be more sensitive and better aligned with the hypothesis than a purely cluster-mean-based filter.

3. **Refine the clustering step if necessary**

   The current Leiden resolution (1.0) produces 8–10 clusters; within each, there can be internal Purity gradations:

   - Explore a **range of resolutions** (e.g., 0.5, 1.0, 1.5) per population and see whether any settings yield clusters with weaker QC separation but clearer Purity separation.
   - Consider running neighbors on a representation that is **regressed for QC** (sc.pp.regress_out on log10(UMI), Complexity) before clustering, specifically for PD and PP. That might produce clusters where residual biology dominates and QC variation is dampened.

   This would make it more likely to find clusters whose Purity differences are not trivially due to depth.

4. **Plan for the next analysis step (DE and signatures)**

   For the next planned step—DE and Purity signatures—given the current results, I would adjust the design as follows:

   - Define groups based on **residual Purity** rather than raw Purity:
     - After fitting Purity ~ QC + Sample_ID, take residuals and split into top/bottom quartiles.
   - Perform DE within each population (PD and PP) comparing high vs low **residual Purity** cells, not just raw Purity.
   - If you still want to leverage subclusters:
     - For each subcluster, compute mean residual Purity; treat clusters with high vs low residual Purity as “Purity-enriched/depleted” for DE between clusters.
     - This is more faithful to “Purity independent of QC” than the current threshold rule.

   When you subsequently fit linear models relating **signature scores** to Purity while adjusting for QC and Sample_ID, pay close attention to:

   - Whether Purity coefficients remain significant after QC adjustment.
   - Whether the model using **residual Purity** better explains signature variation than QC alone.

   That will give a clearer yes/no readout on the hypothesis.

5. **Interpretation of current evidence vs hypothesis**

   - The KW tests confirm that PD and PP contain subclusters with strong differences in Purity *and* QC. So far, **Purity is not separable from QC at the cluster level** under your strict criteria.
   - The absence of any flagged clusters with both substantial Purity shifts and small QC shifts argues **against** a naïve interpretation where Purity captures purely biological microenvironmental states that are entirely independent of technical depth.
   - However, given the granularity of clustering and the strict thresholds, the more nuanced question—whether a **residual Purity component** (after adjusting for QC) has distinct transcriptional correlates—remains open and should be addressed with cell-level modeling and residual-based group definitions in the next step.

In summary: this step shows strong substructure but also strong coupling between Purity and QC at the cluster level, so the hypothesis is not supported yet. The next steps should explicitly factor QC into the modeling (via regression and residuals) and potentially re-define Purity-enriched/depleted groups using residual Purity rather than raw cluster means.

## Next Steps
Step 1: Within PD and PP, jointly perform the full residual-Purity program analysis by regressing Purity on log10(UMI Count), Complexity, and Sample_ID to obtain QC-adjusted Purity residuals, defining high vs low residual-Purity groups, running within-population DE (high vs low residual-Purity) to build Purity-up/down signatures, scoring these signatures across all cells, and finally regressing signature scores on raw Purity with QC and Sample_ID as covariates to quantify QC-independent Purity-associated transcriptional programs.

## This code identifies transcriptional signatures associated with “residual” purity within specified cell populations after rigorously adjusting for technical and sample-level covariates. It (1) regresses Purity on UMI count, complexity, and sample ID to derive QC-adjusted purity residuals; (2) performs differential expression between cells with high vs low residual purity to define up/down gene signatures; (3) scores these signatures across all cells; and (4) tests how strongly these scores associate with raw purity (again controlling for QC covariates), storing effect sizes and model summaries for downstream interpretation.

In [ ]:

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats

# Populations to analyze (QC-robust, spatially structured)
populations_of_interest = ['PD', 'PP']

# Ensure required columns and dtypes
required_obs = ['Populations', 'Sample_ID', 'Purity', 'UMI Count', 'Complexity']
for col in required_obs:
    if col not in adata.obs.columns:
        raise ValueError(f"Required column '{col}' not found in adata.obs")

adata.obs['Populations'] = adata.obs['Populations'].astype('category')
adata.obs['Sample_ID'] = adata.obs['Sample_ID'].astype('category')
for col in ['Purity', 'UMI Count', 'Complexity']:
    adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce')

# Ensure expression matrix is log-normalized for DE and scoring
# This assumes raw counts are in adata.X; if already log1p-normalized, this will be a no-op aside from a warning.
if adata.raw is None:
    # Create a copy of raw counts before normalization if not present
    adata.raw = adata.copy()

# Heuristic: if the maximum value is large (>20), treat as counts and normalize/log1p
X_max = np.max(adata.X)
if X_max > 20:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

# Helper: fit OLS with design matrix X and response y, return coefficients and residuals
def fit_ols(X, y):
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    # Remove rows with NaN
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X = X[mask]
    y = y[mask]
    if X.shape[0] < X.shape[1] + 1:
        raise ValueError("Not enough valid observations to fit OLS model.")
    # Add intercept
    X_design = np.column_stack([np.ones(X.shape[0]), X])
    # Solve via least squares
    beta, _, _, _ = np.linalg.lstsq(X_design, y, rcond=None)
    y_hat = X_design @ beta
    residuals = y - y_hat
    return beta, residuals, mask

# Container for regression summaries
regression_summaries = []

for pop in populations_of_interest:
    pop_mask = adata.obs['Populations'] == pop
    n_pop = int(pop_mask.sum())
    if n_pop < 500:
        print(f"Population {pop}: only {n_pop} cells; skipping residual-Purity analysis due to low power.")
        continue

    print(f"\n=== QC-adjusted Purity and residual-based DE for population {pop} (n={n_pop}) ===")

    obs_pop = adata.obs.loc[pop_mask, ['Purity', 'UMI Count', 'Complexity', 'Sample_ID']].copy()

    # Build design matrix for Purity ~ log10(UMI) + Complexity + Sample dummies
    log_umi = np.log10(obs_pop['UMI Count'].values + 1.0)
    complexity = obs_pop['Complexity'].values
    # One-hot encode Sample_ID (drop first to avoid collinearity)
    sample_dummies = pd.get_dummies(obs_pop['Sample_ID'], drop_first=True)
    X_qc = np.column_stack([log_umi, complexity, sample_dummies.values])
    y_purity = obs_pop['Purity'].values

    beta, residuals, valid_mask = fit_ols(X_qc, y_purity)

    # Store residual Purity back to the main AnnData with explicit index alignment
    resid_series = pd.Series(np.nan, index=obs_pop.index)
    resid_series.loc[obs_pop.index[valid_mask]] = residuals
    adata.obs.loc[pop_mask, f'Purity_residual_{pop}'] = resid_series

    # Summarize QC-explained variance
    y_valid = y_purity[valid_mask]
    sse = np.sum(residuals ** 2)
    sst = np.sum((y_valid - y_valid.mean()) ** 2)
    r2 = 1.0 - sse / sst if sst > 0 else np.nan
    print(f"Population {pop}: R^2 of Purity explained by log10(UMI), Complexity, and Sample_ID = {r2:.3f}")

    # Define high vs low residual-Purity groups (top/bottom quartiles among valid cells)
    resid_vals = resid_series.values
    valid_resid = np.isfinite(resid_vals)
    if valid_resid.sum() < 200:
        print(f"Population {pop}: fewer than 200 cells with valid residual Purity; skipping DE.")
        continue

    resid_in_pop = resid_vals[valid_resid]
    q25, q75 = np.percentile(resid_in_pop, [25, 75])
    high_mask_local = (resid_vals >= q75) & valid_resid
    low_mask_local = (resid_vals <= q25) & valid_resid

    n_high = int(high_mask_local.sum())
    n_low = int(low_mask_local.sum())
    print(f"Population {pop}: high-residual Purity n={n_high}, low-residual Purity n={n_low}")
    if n_high < 100 or n_low < 100:
        print(f"Population {pop}: insufficient cells in high/low residual-Purity groups; skipping DE.")
        continue

    # Create a temporary group label for DE: 'high_resid', 'low_resid', 'other'
    group_label = pd.Series('other', index=adata.obs.index)
    # Use index alignment instead of boolean & between different-length arrays
    group_label.loc[obs_pop.index[high_mask_local[valid_resid]]] = 'high_resid'
    group_label.loc[obs_pop.index[low_mask_local[valid_resid]]] = 'low_resid'
    adata.obs[f'resid_purity_group_{pop}'] = group_label.astype('category')

    # Subset AnnData to this population for DE
    adata_pop = adata[pop_mask].copy()

    # Map group labels into the subset
    adata_pop.obs['resid_group'] = adata.obs.loc[pop_mask, f'resid_purity_group_{pop}'].astype('category').values

    # Differential expression: high_resid vs low_resid within this population
    sc.tl.rank_genes_groups(
        adata_pop,
        groupby='resid_group',
        groups=['high_resid'],
        reference='low_resid',
        method='wilcoxon'
    )

    # Extract DE table for high_resid vs low_resid
    de = sc.get.rank_genes_groups_df(adata_pop, group='high_resid')

    # Compute BH-FDR manually on the pvals
    pvals = de['pvals'].values.astype(float)
    m = pvals.size
    order = np.argsort(pvals)
    ranked = pvals[order]
    bh = ranked * m / (np.arange(m) + 1)
    bh = np.minimum.accumulate(bh[::-1])[::-1]
    fdr = np.empty_like(bh)
    fdr[order] = bh
    de['pvals_adj'] = fdr

    # Define Purity-up and Purity-down signatures using effect size and FDR cutoffs
    up_genes = de[(de['logfoldchanges'] > 0.25) & (de['pvals_adj'] < 0.05)]['names'].tolist()
    down_genes = de[(de['logfoldchanges'] < -0.25) & (de['pvals_adj'] < 0.05)]['names'].tolist()

    print(f"Population {pop}: {len(up_genes)} Purity-up genes, {len(down_genes)} Purity-down genes (QC-adjusted).")

    # Store signatures in adata.uns for reuse
    adata.uns.setdefault('purity_residual_signatures', {})[pop] = {
        'up_genes': up_genes,
        'down_genes': down_genes,
    }

    # Score genes in the full dataset; explicitly use the normalized/log1p X, not .raw
    if len(up_genes) > 0:
        sc.tl.score_genes(adata, gene_list=up_genes, score_name=f'Purity_up_score_{pop}', use_raw=False)
    else:
        adata.obs[f'Purity_up_score_{pop}'] = np.nan

    if len(down_genes) > 0:
        sc.tl.score_genes(adata, gene_list=down_genes, score_name=f'Purity_down_score_{pop}', use_raw=False)
    else:
        adata.obs[f'Purity_down_score_{pop}'] = np.nan

    # Now, within this population, regress signature scores on raw Purity with QC covariates
    for score_col in [f'Purity_up_score_{pop}', f'Purity_down_score_{pop}']:
        if score_col not in adata.obs.columns:
            continue
        obs_sc = adata.obs.loc[pop_mask, ['Purity', 'UMI Count', 'Complexity', 'Sample_ID', score_col]].copy()
        y = pd.to_numeric(obs_sc[score_col], errors='coerce').values
        if np.sum(np.isfinite(y)) < 200:
            print(f"Population {pop}, score {score_col}: insufficient valid scores for regression; skipping.")
            continue

        log_umi_sc = np.log10(obs_sc['UMI Count'].values + 1.0)
        complexity_sc = obs_sc['Complexity'].values
        sample_dummies_sc = pd.get_dummies(obs_sc['Sample_ID'], drop_first=True)
        purity_sc = obs_sc['Purity'].values
        X_sc = np.column_stack([purity_sc, log_umi_sc, complexity_sc, sample_dummies_sc.values])

        beta_sc, resid_sc, valid_sc = fit_ols(X_sc, y)

        # Compute R^2 for this model
        y_valid_sc = y[valid_sc]
        sse_sc = np.sum(resid_sc ** 2)
        sst_sc = np.sum((y_valid_sc - y_valid_sc.mean()) ** 2)
        r2_sc = 1.0 - sse_sc / sst_sc if sst_sc > 0 else np.nan

        # Construct design matrix (with intercept) for SE/p-value computation
        X_sc_valid = np.column_stack([np.ones(X_sc[valid_sc].shape[0]), X_sc[valid_sc]])
        n_obs, n_param = X_sc_valid.shape
        try:
            XtX = X_sc_valid.T @ X_sc_valid
            XtX_inv = np.linalg.inv(XtX)
        except np.linalg.LinAlgError:
            print(f"Population {pop}, score {score_col}: singular X'X matrix; skipping SE/p-value computation.")
            regression_summaries.append({
                'population': pop,
                'score': score_col,
                'beta_purity': beta_sc[1] if len(beta_sc) > 1 else np.nan,
                'se_purity': np.nan,
                't_purity': np.nan,
                'p_purity': np.nan,
                'r2_model': r2_sc,
                'n_obs': int(n_obs),
                'n_params': int(n_param),
            })
            continue

        sigma2 = sse_sc / max(n_obs - n_param, 1)
        var_beta = np.diag(XtX_inv) * sigma2
        se_beta = np.sqrt(np.maximum(var_beta, 1e-12))
        # beta_sc includes intercept at index 0; Purity is first covariate at index 1
        beta_purity = beta_sc[1]
        se_purity = se_beta[1]
        t_purity = beta_purity / se_purity if se_purity > 0 else np.nan
        p_purity = 2 * (1 - stats.norm.cdf(abs(t_purity))) if np.isfinite(t_purity) else np.nan

        print(f"Population {pop}, score {score_col}: beta_Purity={beta_purity:.4f}, SE={se_purity:.4f}, t={t_purity:.2f}, p={p_purity:.3e}, R^2={r2_sc:.3f}")

        regression_summaries.append({
            'population': pop,
            'score': score_col,
            'beta_purity': beta_purity,
            'se_purity': se_purity,
            't_purity': t_purity,
            'p_purity': p_purity,
            'r2_model': r2_sc,
            'n_obs': int(n_obs),
            'n_params': int(n_param),
        })

# Store regression summaries in adata.uns for later inspection
if regression_summaries:
    adata.uns['purity_residual_signature_regression'] = pd.DataFrame(regression_summaries)
    print("\nStored regression summaries in adata.uns['purity_residual_signature_regression'].")
else:
    print("\nNo regression summaries were generated (likely due to missing signatures or insufficient cells).")



=== QC-adjusted Purity and residual-based DE for population PD (n=16624) ===
Population PD: R^2 of Purity explained by log10(UMI), Complexity, and Sample_ID = 0.078
Population PD: high-residual Purity n=4156, low-residual Purity n=4156
ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


Population PD: 39 Purity-up genes, 11 Purity-down genes (QC-adjusted).
computing score 'Purity_up_score_PD'


    finished: added
    'Purity_up_score_PD', score of gene set (adata.obs).
    141 total control genes are used. (0:00:00)


computing score 'Purity_down_score_PD'


...

Population PP: 27 Purity-up genes, 14 Purity-down genes (QC-adjusted).
computing score 'Purity_up_score_PP'


    finished: added
    'Purity_up_score_PP', score of gene set (adata.obs).
    123 total control genes are used. (0:00:00)


computing score 'Purity_down_score_PP'


    finished: added
    'Purity_down_score_PP', score of gene set (adata.obs).
    75 total control genes are used. (0:00:00)


Population PP, score Purity_up_score_PP: beta_Purity=0.4784, SE=0.0186, t=25.70, p=0.000e+00, R^2=0.127
Population PP, score Purity_down_score_PP: beta_Purity=-0.4827, SE=0.0224, t=-21.54, p=0.000e+00, R^2=0.129

Stored regression summaries in adata.uns['purity_residual_signature_regression'].


### Agent Interpretation

The results support the hypothesis that, within PD and PP, there are transcriptional programs tied to a Purity-like microenvironmental signal that are not reducible to UMI depth, complexity, or sample effects.

Key points from the output:

1. **QC covariates only weakly explain Purity within PD and PP**  
   - PD: R² = 0.078 for Purity ~ log10(UMI) + Complexity + Sample_ID  
   - PP: R² = 0.123 for the same model  
   This leaves substantial residual variation in Purity after QC and sample are accounted for, so there is room for a genuine microenvironmental component.

2. **Residual-based DE finds robust Purity-linked signatures within each population**  
   - PD: 39 residual-Purity-up and 11 residual-Purity-down genes  
   - PP: 27 up and 14 down  
   These are defined strictly within-population, using residual Purity and controlling for QC/sample via the residualization step. This is conceptually well-aligned with the hypothesis.

3. **Residual-Purity programs remain strongly associated with raw Purity after QC adjustment**  
   In the regression of signature scores on raw Purity plus QC covariates:
   - PD:
     - Purity_up_score_PD: β_Purity = +0.34, t ≈ 38.7, R² = 0.088  
     - Purity_down_score_PD: β_Purity = −0.50, t ≈ −33.9, R² = 0.078  
   - PP:
     - Purity_up_score_PP: β_Purity = +0.48, t ≈ 25.7, R² = 0.127  
     - Purity_down_score_PP: β_Purity = −0.48, t ≈ −21.5, R² = 0.129  

   Effect sizes are sizable and highly significant even after including log10(UMI), Complexity, and Sample_ID. This is exactly the pattern the hypothesis anticipates: residual-defined programs that still track raw Purity in a QC-adjusted model.

Taken together, these results are **strongly consistent** with the hypothesis: in both PD and PP there are QC-adjusted, Purity-linked expression programs that reflect something beyond generic technical or sample-level effects.

To strengthen and extend this analysis in distinct ways going forward:

1. **Inspect gene content and overlap of PD vs PP residual-Purity signatures**  
   - Check overlap between PD and PP up-genes and between down-genes.  
     - If there is substantial overlap, that suggests a **shared, generalized microenvironmental purity axis** across related populations.  
     - If they’re largely distinct, it implies **population-specific Purity responses**.
   - Within each population, characterize whether up vs down genes cluster into interpretable modules: e.g. matrix/adhesion vs cell-cycle vs signaling, etc., using only within-dataset co-expression (no external databases):  
     - Correlation clustering among up-genes or down-genes.  
     - Module-level scoring and comparing modules’ Purity associations.

2. **Check residual-Purity signal is really not a hidden QC artifact**  
   Even though you explicitly controlled for UMI Count and Complexity, verify that:
   - Residual Purity is only weakly correlated with these QC metrics within PD/PP (e.g., Pearson/Spearman r between residual and log10(UMI), Complexity).  
   - Signature scores (up/down) are only weakly associated with QC within population when Purity is not included in the model (regress scores on QC+Sample_ID only, to show βs are small).  
   This will further rule out more subtle QC structure.

3. **Relate residual-Purity programs to spatial organization (new angle)**  
   To keep distinct from the paper and previous analysis, lean into spatial patterning of these residual programs:
   - Plot spatial maps of Purity_up_score_PD/PP and Purity_down_score_PD/PP and see whether high residual-Purity programs localize to particular anatomical regions or interfaces.  
   - Within PD or PP, correlate residual Purity (and the signature scores) with local neighborhood composition (e.g., fraction of neighboring cell types within a spatial radius) to see whether the residual Purity signal captures specific **local microenvironmental mixtures**.

4. **Assess within-population heterogeneity of the Purity programs**  
   - Use the Purity_up/down scores as features for within-PD and within-PP subclustering or ordering (e.g., UMAP with these scores plus a few other top genes).  
   - Test whether subclusters differ systematically in residual Purity and in proximity to certain other labeled populations in space, indicating **functional states within a population** tied to microenvironmental purity.

5. **Cross-sample robustness of the residual signatures**  
   - Evaluate, within each population, the Purity–signature relationship per Sample_ID:  
     - Refit the score ~ Purity + QC model separately in each sample, or include a Purity×Sample interaction.  
     - Consistent sign and magnitude across samples would argue that this is a robust microenvironmental axis rather than a single-sample artifact.
   - Optionally, compute a mixed-effects version conceptually (Sample_ID as random effect) by approximating: compare per-sample β_Purity distributions.

6. **Directionality and co-variation of up vs down signatures**  
   - Within PD or PP, check the correlation between Purity_up and Purity_down scores. Strong negative correlation would indicate a **single underlying axis**; weaker or more complex relationships might imply multiple independent Purity-related states.
   - Visualize both scores on the same spatial embedding to see whether up and down signatures are mutually exclusive regions or interdigitated.

7. **Quantify how much Purity variance these programs capture**  
   You’ve measured R² of score ~ Purity + QC. To tie this back to Purity itself:
   - Regress Purity on the up and down scores plus QC within each population: Purity ~ score_up + score_down + QC + Sample_ID.  
   - Compare R² with and without the scores. The increment in R² would quantify how much of the **residual Purity variation** is explained by these transcriptional programs.

In summary, the current step gives strong evidence that PD and PP harbor QC-adjusted, microenvironmental Purity programs that continue to track raw Purity after controlling for QC and sample effects. Next, you should (i) unpack gene content and shared vs distinct programs across PD/PP, (ii) anchor these programs spatially and in neighborhood context, and (iii) dissect within-population heterogeneity and cross-sample robustness to fully characterize these microenvironmental axes in a way that is clearly distinct from prior analyses.